# XGBoost training and evaluation

Train a next-hour BTC direction classifier, calibrate its probabilities, and evaluate predictions selected by model confidence.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.special import logit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

In [2]:
features = pd.read_parquet(ROOT / "data" / "features.parquet")

FEATURE_COLUMNS = [
    column
    for column in features.columns
    if column not in {"timestamp", "future_btc_return_1h", "target"}
]

## Chronological split

Use the first 70% of observations for training, the next 10% for probability calibration and threshold selection, and the final 20% for testing. All timestamps are UTC.

In [3]:
n = len(features)
train_end, calibration_end = int(n * 0.7), int(n * 0.8)

splits = {
    "train": features.iloc[:train_end],
    "calibration": features.iloc[train_end:calibration_end],
    "test": features.iloc[calibration_end:],
}

pd.DataFrame(
    {
        "hours": [len(frame) for frame in splits.values()],
        "start_utc": [frame["timestamp"].min().strftime("%Y-%m-%d %H:%M") for frame in splits.values()],
        "end_utc": [frame["timestamp"].max().strftime("%Y-%m-%d %H:%M") for frame in splits.values()],
    },
    index=list(splits),
)

,hours,start_utc,end_utc
train,3108,2025-08-09 05:00,2025-12-16 16:00
calibration,444,2025-12-16 17:00,2026-01-04 08:00
test,888,2026-01-04 09:00,2026-02-10 12:00


In [4]:
X_train = splits["train"][FEATURE_COLUMNS]
y_train = splits["train"]["target"].to_numpy()

X_calibration = splits["calibration"][FEATURE_COLUMNS]
y_calibration = splits["calibration"]["target"].to_numpy()

X_test = splits["test"][FEATURE_COLUMNS]
y_test = splits["test"]["target"].to_numpy()

## Fit XGBoost

A shallow, regularized XGBoost classifier estimates the probability of an up hour from the training observations.

In [5]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=2,
    learning_rate=0.04,
    min_child_weight=30,
    subsample=0.9,
    colsample_bytree=0.8,
    reg_lambda=20,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train);

## Calibrate probabilities

Platt scaling fits a logistic regression to the model's log-odds on the calibration period, adjusting its probability estimates before confidence-based selection.

In [6]:
raw_calibration = model.predict_proba(X_calibration)[:, 1]
raw_test = model.predict_proba(X_test)[:, 1]

calibration_log_odds = logit(
    np.clip(raw_calibration, 1e-6, 1 - 1e-6)
).reshape(-1, 1)
test_log_odds = logit(
    np.clip(raw_test, 1e-6, 1 - 1e-6)
).reshape(-1, 1)

In [7]:
platt = LogisticRegression(C=1_000_000, max_iter=2000)
platt.fit(calibration_log_odds, y_calibration)

p_calibration = platt.predict_proba(calibration_log_odds)[:, 1]
p_test = platt.predict_proba(test_log_odds)[:, 1]

## Select the confidence threshold

For an estimated up probability $p$, confidence is $\max(p, 1-p)$, the probability assigned to the predicted direction. Choose the threshold with the highest calibration accuracy among those retaining at least 40 predictions, then fix it for test evaluation.

In [8]:
MIN_CALIBRATION_HOURS = 40

calibration_predicted = (p_calibration >= 0.5).astype(int)
calibration_confidence = np.maximum(p_calibration, 1 - p_calibration)

rows = []
for threshold in np.round(np.arange(0.50, 0.60, 0.01), 2):
    mask = calibration_confidence >= threshold
    if mask.sum() < MIN_CALIBRATION_HOURS:
        continue

    rows.append(
        {
            "threshold": threshold,
            "hours": int(mask.sum()),
            "accuracy": accuracy_score(y_calibration[mask], calibration_predicted[mask]),
        }
    )

In [9]:
calibration_thresholds = pd.DataFrame(rows).sort_values(
    ["accuracy", "hours"], ascending=False
)
CONFIDENCE_THRESHOLD = float(calibration_thresholds.iloc[0]["threshold"])

print(f"Selected confidence threshold: {CONFIDENCE_THRESHOLD:.2f}")

Selected confidence threshold: 0.53


## Test accuracy

Apply the selected threshold to the test period and measure directional accuracy on the retained predictions.

In [10]:
predicted = (p_test >= 0.5).astype(int)
test_confidence = np.maximum(p_test, 1 - p_test)

selected = test_confidence >= CONFIDENCE_THRESHOLD
selected_accuracy = accuracy_score(y_test[selected], predicted[selected])

print(f"High-confidence test accuracy: {selected_accuracy:.2%}")

High-confidence test accuracy: 55.08%
